# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

# 📥 Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

## ✅ Task 1 & 3 Applied: Deeper ANN + More EpochsWe added extra Dense layers (1024 → 512 → 256 → 128) and increased epochs to **20** soyou can see whether a bigger, deeper ANN closes the gap with CNN (spoiler: it won't,because it still has no way to understand spatial structure).

In [ ]:
ann_model = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Test Accuracy:", ann_test_acc)

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

## ✅ Task 2 & 4 Applied: Filter Progression + EarlyStoppingThe CNN already uses the requested filter progression **32 → 64 → 128**. We added a`BatchNormalization` after the last conv layer for consistency, bumped epochs to **20**,and added `EarlyStopping` (patience=3, restores best weights) so training stopsautomatically once validation loss stops improving — this prevents wasted epochs andoverfitting.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print("CNN Test Accuracy:", cnn_test_acc)

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(ann_history.history['val_accuracy'], label='ANN Val Acc')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc')
plt.plot(aug_history.history['val_accuracy'], label='CNN+Aug Val Acc')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN vs Augmented CNN Validation Accuracy")
plt.legend()
plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

## ✅ Task 5 Applied: Data Augmentation Actually TrainedInstead of leaving the augmented model defined-but-unused, we now train it for real(20 epochs, with `EarlyStopping`) and evaluate it on the test set. Compare`aug_test_acc` against `cnn_test_acc` below to see whether augmentation improvedgeneralization on CIFAR-10.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

aug_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Task 5: actually train with augmentation enabled
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[aug_early_stop]
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test)
print("Augmented CNN Test Accuracy:", aug_test_acc)

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN (deeper)", "CNN", "CNN + Augmentation"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc, aug_test_acc]
})
comparison.sort_values("Test Accuracy", ascending=False)

# 🎓 Student Learning Tasks

### ✅ Beginner Tasks (all applied in this notebook)
1. ~~Increase ANN layers and observe performance~~ → done (1024→512→256→128)
2. ~~Change CNN filters from 32→64→128~~ → done
3. ~~Increase epochs to 20~~ → done for all three models
4. ~~Add EarlyStopping~~ → done (patience=3, restores best weights)
5. ~~Add data augmentation training~~ → done, with its own test accuracy reported

### 🔥 Intermediate Tasks (try these next)
6. Add `ReduceLROnPlateau` to shrink the learning rate when validation loss plateaus
7. Try `Conv2D` with `padding='same'` to preserve spatial dimensions longer
8. Add a fourth convolutional block and see if accuracy keeps improving or starts to overfit
9. Swap `BatchNormalization` placement (before vs after activation) and compare
10. Plot a confusion matrix on the CNN's test predictions to see which classes it confuses most (e.g., cat vs dog)

### 🚀 Advanced Tasks
11. Fine-tune a pretrained model (e.g., `MobileNetV2` or `ResNet50` with `include_top=False`) via transfer learning
12. Implement mixup or cutout augmentation
13. Try a learning-rate warmup + cosine decay schedule
14. Export the best model with `model.save()` and write an inference script that classifies a new image

# ✅ Conclusion
- **ANN works**, but ignores image structure — even a much deeper ANN (1024→512→256→128)
  plateaus well below the CNN because flattening a 32×32×3 image destroys spatial relationships.
- **CNN extracts spatial features** via convolution + pooling, so it performs significantly
  better with far fewer parameters than a comparably deep ANN.
- **Data augmentation** (random flip/rotation/zoom) generally improves generalization,
  though the exact gain on CIFAR-10 depends on how long you train and how strong the
  augmentation is — check the comparison table above for this run's numbers.
- **EarlyStopping** saved training time by stopping once validation loss stopped improving,
  and `restore_best_weights=True` ensures the final model is the best checkpoint, not
  just the last one.
- This project builds strong fundamentals for **computer vision interviews and deep
  learning projects** — the next step is transfer learning with a pretrained backbone
  (see Advanced Tasks above).